In [90]:
import numpy as np
import pandas as pd

initial_df = pd.read_excel("data_clear.xlsx", parse_dates=["event_time"])

## Обрабатываем изначальный DataFrame, чтобы работать только с заданным промежутком времени

In [119]:
# 24.09.2020 - 28.02.2021
def set_time_intervals(start: str, end: str, df: pd.DataFrame) -> pd.DataFrame:
    start = list(map(int, start.split(".")))
    end = list(map(int, end.split(".")))
    start_ts = pd.Timestamp(day=start[0], month=start[1], year=start[2], tz="UTC")
    end_ts = pd.Timestamp(day=end[0], month=end[1], year=end[2], tz="UTC") + pd.Timedelta(days=1)
    time_mask = (start_ts <= df["event_time"]) & (end_ts > df["event_time"]) 
    return df[time_mask]


df = set_time_intervals("24.09.2020", "30.09.2020", initial_df)

## 1). Анализ пользователей, процентное соотношение всех событий к количеству сессий

In [138]:
from typing import Optional

class EventTypeAnalysis:
    def __init__(self, df: pd.DataFrame):
        self.LABEL = "Анализ совершённых событий пользователями"
        self.df: Optional[pd.DataFrame] = None
        self.total_sessions = len(df)

        self.views = df["has_view"].sum()
        self.carts = df["has_cart"].sum()
        self.purchases = df["has_purchase"].sum()

        self.view_percentage = round(self.views / self.total_sessions, 3)
        self.cart_percentage = round(self.carts / self.total_sessions, 3)
        self.purchases_percentage = round(self.purchases / self.total_sessions, 3)

    def make_analysis(self) -> pd.DataFrame:
        self.df = pd.DataFrame(
            columns=[self.LABEL],
            index=[
                "Колличество сессий",
                "Просмотры",
                "Добавлений в карзину",
                "Покупок",
                "% просмотров",
                "% добавлений в карзину",
                "% покупок"
                ])
        self.df.loc["Колличество сессий", self.LABEL] = self.total_sessions
        self.df.loc["Просмотры", self.LABEL] = self.views
        self.df.loc["Добавлений в карзину", self.LABEL] = self.carts
        self.df.loc["Покупок", self.LABEL] = self.purchases
        self.df.loc["% просмотров", self.LABEL] = self.view_percentage
        self.df.loc["% добавлений в карзину", self.LABEL] = self.cart_percentage
        self.df.loc["% покупок", self.LABEL] = self.purchases_percentage
        return self.df

event_type_df = df.groupby(["user_session"])["event_type"].apply(set).reset_index()
event_type_df["has_view"] = event_type_df["event_type"].apply(lambda x: "view" in x)
event_type_df["has_cart"] = event_type_df["event_type"].apply(lambda x: "cart" in x)
event_type_df["has_purchase"] = event_type_df["event_type"].apply(lambda x: "purchase" in x)

event_type_analysis = EventTypeAnalysis(event_type_df)
event_type_analysis.make_analysis()
# event_type_df[event_type_df["has_view"] == False]


,Анализ совершённых событий пользователями
Колличество сессий,16990
Просмотры,16935
Добавлений в карзину,1119
Покупок,667
% просмотров,0.997
% добавлений в карзину,0.066
% покупок,0.039


## 2). Анализ времени между событиями, сколько пользователь тратит времени между просмотром товара, добавлением в корзину и покупкой

In [140]:
df.sort_values(["user_session", "event_time"])

,event_time,event_type,product_id,category_code,brand,price,user_id,user_session
27192,2020-09-30 17:52:02+00:00,view,1675002,electronics.tablet,xiaomi,36646.14,1515915625509230080,000c34fa-991f-442a-8e07-8c472269bec6
9364,2020-09-26 18:59:06+00:00,view,3582066,computers.components.tv_tuner,d-color,2431.65,1515915625520059904,002DmERG1w
8955,2020-09-26 16:55:42+00:00,view,1247635,electronics.telephone,sirius,3645.30,1515915625519680000,009so5Mgnj
2481,2020-09-25 03:44:48+00:00,view,1756713,electronics.telephone,NaN,1822.65,1515915625519579904,00scna0OEj
2487,2020-09-25 03:46:21+00:00,view,1429384,electronics.telephone,NaN,696.00,1515915625519579904,00scna0OEj
...,...,...,...,...,...,...,...,...
22989,2020-09-29 19:55:01+00:00,view,275243,NaN,tp-link,709.92,1515915625520950016,zzrmGmDaIC
1130,2020-09-24 16:29:43+00:00,view,957592,electronics.tablet,samsung,1223.22,1515915625519480064,zzt9kKgXen
25849,2020-09-30 12:31:41+00:00,view,893196,computers.components.videocards,sapphire,18626.70,1515915625521110016,zztRHEozV0
25852,2020-09-30 12:32:12+00:00,view,877060,computers.components.videocards,msi,16294.23,1515915625521110016,zztRHEozV0


In [139]:
def func(df):
    print(df)
    print(type(df))
    print("--------------------------------")

df.sort_values(["user_session", "event_time"])
df.groupby("user_session").apply(func)

                     event_time event_type  product_id       category_code  \
27192 2020-09-30 17:52:02+00:00       view     1675002  electronics.tablet   

        brand     price              user_id  
27192  xiaomi  36646.14  1515915625509230080  
<class 'pandas.DataFrame'>
--------------------------------
                    event_time event_type  product_id  \
9364 2020-09-26 18:59:06+00:00       view     3582066   

                      category_code    brand    price              user_id  
9364  computers.components.tv_tuner  d-color  2431.65  1515915625520059904  
<class 'pandas.DataFrame'>
--------------------------------
                    event_time event_type  product_id          category_code  \
8955 2020-09-26 16:55:42+00:00       view     1247635  electronics.telephone   

       brand   price              user_id  
8955  sirius  3645.3  1515915625519680000  
<class 'pandas.DataFrame'>
--------------------------------
                    event_time event_type  product_

KeyboardInterrupt: 